# TP53 Mutation Predictor — Final Report

**Predicting TP53 mutation status and mutation type from bulk RNA-seq, with biologically-informed graph priors and external validation on TCGA primary tumours.**

---

## Abstract

*TP53* is the most frequently mutated gene in human cancer. We built a fully reproducible pipeline that predicts **(Task 1) TP53 mutation status (binary)** and **(Task 2) TP53 mutation type (multiclass)** from bulk RNA-seq expression profiles, using the Cancer Cell Line Encyclopedia (CCLE / DepMap 24Q4) for training and the TCGA pan-cancer atlas (n = 8,424 primary tumours) for external validation.

Across nine model variants spanning gradient boosting (XGBoost), logistic regression, graph convolutional networks (GCN) on five graph topologies (Spearman co-expression at three thresholds, STRING physical PPI, hybrid bio + co-expression), and graph attention networks (GAT), we find:

1. **XGBoost on top-2,000 highly variable genes is the strongest model** (CCLE OOF F1 = 0.875, ROC-AUC = 0.906) — matching the published bulk benchmark from Ravasio (2024).
2. **The model independently rediscovers canonical TP53 biology**: SHAP analysis identifies CDKN1A (p21) as the dominant predictor (mean |SHAP| = 1.27, three-fold larger than the next gene), and a hypergeometric test shows TP53 pathway genes are enriched among top-K SHAP features at **52.6× (p = 1.07 × 10⁻⁸)** for K=10, dropping monotonically with K.
3. **CCLE → TCGA transfer is real for XGBoost** (ROC-AUC = 0.806 with per-cohort z-score; AUC > 0.85 on 8 of 31 cancer types); but **vanilla GCN/GAT models do not transfer** under the same protocol (TCGA AUC ≈ 0.40, model collapses), an honest negative result attributable to BatchNorm distribution shift and raw-feature scale mismatch.
4. **GAT on a dense hybrid graph is the best GNN** (F1 = 0.760), but every graph-based model trails XGBoost by ~0.12 F1 on within-cohort metrics — biological graph priors do not surpass HVG + tabular learning at this scale.
5. **Mutation-type classification (Task 2) is much harder than mutation-status classification (Task 1)**, consistent with the loss-of-function biology: every TP53-mutant cell line loses p53 transcriptional activity in similar ways, so the bulk transcriptome cannot easily distinguish how the gene was hit.

All code, SLURM job scripts, metric tables, OOF predictions and figures are versioned in this repository. Sections 4–5 below regenerate every table and figure directly from `data/processed/`.

## 1. Background & motivation

*TP53* ("guardian of the genome") sits at the apex of the cellular stress response. When activated by DNA damage, oncogene signalling, or hypoxia, p53 induces a transcriptional program that either arrests the cell cycle (p21 / *CDKN1A*), drives apoptosis (PUMA / *BBC3*, *BAX*, *FAS*), or initiates DNA-repair / senescence cascades. Loss of TP53 function eliminates these brakes and is observed in about half of all human cancers — making it both the most frequently mutated gene in cancer and one of the most pharmacologically un-targetable.

Detecting TP53 mutation status from bulk transcriptomic profiles alone is biologically interesting because TP53 loss leaves a measurable downstream signature (loss of p21 induction, altered apoptosis programs, etc.). It is also a controlled benchmark for two open machine-learning questions:

1. Do **biologically-informed graph priors** (PPI networks, pathway membership) outperform purely statistical ones (Spearman co-expression) when used as the topology for graph neural networks?
2. Do **graph neural networks generalise across cell-line and primary-tumour cohorts** as well as gradient-boosted trees do?

This project answers both questions with explicit comparisons, full metrics, statistical tests, and per-cancer-type breakdowns on TCGA. It also addresses the requested second task: predicting **mutation type** (missense vs truncating vs other) within the TP53-mutant subset.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display, Markdown

PROC = Path('..') / 'data' / 'processed'
PLOTS = PROC / 'plots'
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Data

| Cohort | Source | Samples | Genes | TP53 mutant rate |
|---|---|---:|---:|---:|
| **CCLE** (training) | DepMap 24Q4 — `OmicsExpressionProteinCodingGenesTPMLogp1.csv`, `OmicsSomaticMutations.csv` | 1,673 cell lines | 19,193 protein-coding | **58.9 %** (986 / 1,673) |
| **TCGA** (external) | UCSC Xena PanCancer Atlas — `EB++AdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.xena`, `mc3.v0.2.8.PUBLIC.nonsilentGene.xena` | 8,424 primary tumours | 1,851 (overlap with CCLE top-2k) | **36.5 %** |

TP53 mutation labels are derived from `OmicsSomaticMutations.csv` (CCLE) and the MC3 binary matrix (TCGA) — see `src/load_data.py` and `src/tcga_load.py` for exact code. CCLE labels are split by `MolecularConsequence` into Missense, Frame_Shift_Del, Splice_Site, and Other (kept for Task 2).

The 22 percentage-point gap in TP53 prevalence between cohorts reflects the well-documented **selection bias from immortalisation** — TP53-mutant clones are over-represented in cell lines.

Per-cancer-type TP53 prevalence in TCGA spans the full biological range and recapitulates the textbook landscape: squamous and serous carcinomas dominate, pheochromocytoma and thyroid carcinomas rarely use TP53 loss as a driver.

In [ ]:
display(Image(str(PLOTS / 'domain' / 'tp53_prevalence_per_type.png')))

## 3. Methods (summary)

**Feature selection** — top 2,000 highly variable genes by variance over CCLE, kept fixed across all experiments for direct comparability.

**Cross-validation** — 5-fold stratified CV with `random_state=42`. Fold assignments are computed once (`data/processed/cv_splits.csv`) and reused by every model (XGBoost, GCN, GAT) so that within-cohort comparisons are exactly aligned.

**Within-fold preprocessing** — per-fold z-score statistics for GNN node features (no leakage). XGBoost takes raw log2(TPM+1).

**Graphs (Task 1 GNNs)**:

| Graph | Construction | Edges (undirected) | Avg degree |
|---|---|---:|---:|
| `gene_graph_thr05.npz` | Spearman \|ρ\| ≥ 0.5 | 59,701 | 59.7 |
| `gene_graph_thr07.npz` | Spearman \|ρ\| ≥ 0.7 | 2,556 | 2.6 |
| `gene_graph_topk10.npz` | top-10 per gene by abs ρ | 18,131 | 18.1 |
| `gene_graph_bio.npz` | STRING physical PPI (combined_score ≥ 700) | 1,851 | 1.85 |
| `gene_graph_hybrid.npz` | union (bio ∪ thr=0.5) | 61,183 | 61.2 |

STRING physical PPI and Spearman co-expression are **nearly disjoint** (Jaccard similarity = 0.006 — see `data/processed/graph_overlap.json`), capturing complementary biological signals (physical contact vs co-regulation).

**Models (Task 1 binary)**:
- XGBoost — fixed defaults: `n_estimators=300, max_depth=6, lr=0.05, subsample=0.8, colsample_bytree=0.8, tree_method='hist', objective='binary:logistic'`.
- GCN baseline (v1): 2-layer GCNConv, hidden=64, dropout=0.5, BCE, 100 fixed epochs.
- GCN v2: 3-layer GCNConv, hidden=128, dropout=0.4, BatchNorm + residual, BCE with `pos_weight = N_neg/N_pos`, 2-d node features [raw, z-score], Adam + ReduceLROnPlateau, early stopping on val ROC-AUC (patience 30, max 300 epochs). Trained on all five graphs.
- GAT: 2-layer GATConv with 4 attention heads, otherwise identical to GCN v2 scaffold. Trained on `gene_graph_thr07` and `gene_graph_hybrid`.

**External validation** — final model trained on full CCLE (no CV) is applied to TCGA samples after per-cohort z-score normalisation. 149/2,000 missing genes in TCGA are zero-padded for GNN node features.

**Models (Task 2 multiclass)** — XGBoost (multi:softprob) and multinomial Logistic Regression (L2). 3-class subset of TP53-mutant cell lines: Missense, Truncating (Frame_Shift_Del ∪ Splice_Site, merged because Frame_Shift_Del had only 48 samples), Other.

**Hardware** — Bocconi HPC, partition `stud`, single A100 80 GB. Total compute for the full pipeline: ~6 hours wall-clock.

---
## 4. Task 1 — Binary TP53 mutation status

### 4.1 CCLE within-cohort results (5-fold OOF, n = 1,673)

In [ ]:
tab = pd.read_csv(PROC / 'metrics_table.csv', index_col=0)
tab = tab.sort_values('f1', ascending=False)
tab.style.format('{:.4f}').background_gradient(cmap='Blues', axis=0)

In [ ]:
display(Image(str(PLOTS / 'model_comparison.png')))

**Key observations**

1. **XGBoost dominates every metric** — F1 = 0.875, ROC-AUC = 0.906, PR-AUC = 0.909. Matches the Ravasio (2024) bulk benchmark of F1 ≈ 0.88.
2. **GAT on a dense hybrid graph is the best GNN** (F1 = 0.760, AUC = 0.706) — beats every GCN variant. Attention adds value when the graph has enough neighbours to weight against each other.
3. **GAT on the very-sparse Spearman thr=0.7 graph (2.5k edges) underperforms the GCN baseline** — too few neighbours for attention to be informative.
4. **The choice of graph topology has only small effects on GCN performance** — AUC sits at 0.70–0.71 across thr=0.5, thr=0.7, top-k=10, and hybrid; only the very sparse STRING-only graph (degree 1.85) underperforms.
5. The biological PPI prior did not improve GCN performance (hybrid AUC 0.705 ≈ thr=0.5 AUC 0.701) — the bulk transcriptional signal that XGBoost exploits is mostly per-gene rather than gene-gene relational.

In [ ]:
display(Image(str(PLOTS / 'roc_curves.png')))

### 4.2 Graph stats and overlap

In [ ]:
g = pd.read_csv(PROC / 'graph_stats.csv')
display(g)
with open(PROC / 'graph_overlap.json') as f:
    ov = json.load(f)
display(Markdown(f"**Bio (STRING) vs co-expression overlap** — Jaccard = {ov['jaccard_similarity']:.4f}, intersection = {ov['intersection']:,} edges out of {ov['union']:,} union. Only {100*ov['fraction_bio_in_coexp']:.1f} % of bio edges are also strong co-expression; only {100*ov['fraction_coexp_in_bio']:.1f} % of co-expression edges have direct PPI support. **Two complementary views of \"interaction\"** (physical contact vs shared regulation) — but the union did not improve GCN beyond co-expression alone."))

### 4.3 Threshold calibration on TCGA — why 0.5 is not the right operating point

After CCLE→TCGA transfer, the default threshold of 0.5 leaves the model badly calibrated: it predicts almost every TCGA sample as mutant (predicted positive rate 0.81 vs true positive rate 0.37). The reliability diagram below makes this explicit — TCGA points sit far below the diagonal, meaning predicted probabilities systematically over-estimate the true positive rate.

In [ ]:
display(Image(str(PLOTS / 'calibration_curve.png')))
display(Image(str(PLOTS / 'threshold_calibration.png')))

In [ ]:
thr = pd.read_csv(PROC / 'threshold_calibration_table.csv')
thr.style.format({'threshold': '{:.3f}', 'accuracy': '{:.4f}', 'precision': '{:.4f}',
                  'recall': '{:.4f}', 'f1': '{:.4f}', 'predicted_positive_rate': '{:.4f}'})

**Interpretation** — picking the threshold so the *predicted* TCGA positive rate matches the *actual* TCGA mutation rate (≈ 0.37, achieved at threshold ≈ 0.93 because the model is so over-confident on TCGA) yields the best operating point: **F1 jumps from 0.604 → 0.641, accuracy 0.536 → 0.738**. The F1-optimal threshold derived from CCLE OOF (0.45) does *not* help because it was tuned for CCLE's higher class-prevalence regime — confirming the calibration mismatch is the dominant gap, not discrimination.

### 4.4 Interpretability — SHAP on XGBoost

We train the final XGBoost on all 1,673 CCLE samples and compute per-sample SHAP values via `shap.TreeExplainer`. Top-K SHAP genes are cross-referenced against a curated TP53-pathway gene set (62 HGNC symbols including direct transcriptional targets, regulators, and the TP53 family — see `src/tp53_pathway.py`).

In [ ]:
top20 = pd.read_csv(PROC / 'shap_top20.csv')
top20.style.format({'mean_abs_shap': '{:.4f}', 'mean_signed_shap': '{:+.4f}'}).highlight_max(
    subset=['mean_abs_shap'], color='#fbb4ae')

In [ ]:
display(Image(str(PLOTS / 'shap_top20_pathway.png')))

**Biological reading** — **CDKN1A (p21) towers over everything**, with a mean |SHAP| three times larger than the next gene. p21 is the canonical TP53 transcriptional target; loss of TP53 → loss of p21 induction → uncontrolled proliferation. **The XGBoost has independently learned the textbook TP53 → p21 axis**, with no pathway supervision. Other top features span multiple TP53 effector arms: cell-cycle arrest (CDKN1A, BTG2, CDKN2A), apoptosis (PHLDA3, FAS, TNFRSF10D), cytoskeletal remodelling (CYFIP2).

### 4.5 Formal pathway enrichment (hypergeometric / Fisher's exact)

We test the null hypothesis that the top-K SHAP genes are a uniform-random sample from the 2,000-gene background, against the alternative that they are enriched for the curated TP53-pathway set (19/2,000 of our background genes are pathway members).

In [ ]:
enr = pd.read_csv(PROC / 'shap_enrichment.csv')
enr.style.format({
    'expected_hits': '{:.2f}',
    'enrichment_factor': '{:.2f}x',
    'hypergeom_p': '{:.2e}',
    'fisher_odds_ratio': '{:.1f}',
    'fisher_p': '{:.2e}',
})

In [ ]:
display(Image(str(PLOTS / 'shap_enrichment.png')))

**All three K values are extremely significant** (p ranging 10⁻⁸ to 10⁻¹⁰), with enrichment factors of **52.6× (top-10)**, **36.8× (top-20)**, and **14.7× (top-50)**. This is a formal, statistically-validated rejection of the null — the model's important features are biologically aligned with the canonical TP53 transcriptional program.

### 4.6 TCGA external validation (XGBoost)

In [ ]:
with open(PROC / 'tcga_eval_summary.json') as f:
    s = json.load(f)
rows = []
for label, key in [('CCLE OOF (5-fold, shared 1,851 genes)', 'ccle_oof_5fold_on_shared'),
                    ('TCGA external (CCLE-trained)',          'tcga_external')]:
    m = s[key]
    rows.append({
        'cohort': label, 'n': m['n'],
        'accuracy': m['accuracy'], 'precision': m['precision'],
        'recall': m['recall'], 'f1': m['f1'],
        'roc_auc': m['roc_auc'], 'pr_auc': m['pr_auc'],
        'pos_rate': m['positive_rate'],
    })
tcga_tab = pd.DataFrame(rows)
tcga_tab.style.format({c: '{:.4f}' for c in
    ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'pos_rate']})

In [ ]:
display(Image(str(PLOTS / 'tcga_xgb_roc.png')))

**ROC-AUC drops only ~0.10** from CCLE OOF (0.904) to TCGA (0.806) — a real, modest generalisation gap. Recall *increases* on TCGA (0.97) at threshold 0.5 because the model is over-confident; precision drops correspondingly. After threshold calibration (§ 4.3), TCGA F1 rises to 0.641.

In [ ]:
display(Image(str(PLOTS / 'tcga_xgb_per_cancer_type.png')))
pct = pd.read_csv(PROC / 'tcga_xgb_per_cancer_type.csv').sort_values('roc_auc', ascending=False).head(10)
pct[['cancer_type', 'n', 'positive_rate', 'roc_auc', 'f1']].style.format({
    'positive_rate': '{:.3f}', 'roc_auc': '{:.3f}', 'f1': '{:.3f}'})

**The model performs above 0.85 ROC-AUC on 8 cancer types**, including SKCM (0.97), PCPG (0.94), READ (0.88), ACC (0.86), KICH (0.86), UCEC (0.86), LGG (0.86), LIHC (0.85). Lowest AUCs are on cancers where TP53 is virtually absent (THCA, UVM) or where mutational landscape is dominated by other drivers (PRAD, HNSC) — informative limits, not failures.

### 4.7 GNN cross-cohort failure — honest negative result

We applied the best GCN (v2 thr=0.7) and best GAT (hybrid) to TCGA under the same per-cohort z-score protocol that worked for XGBoost. Both **collapsed to the WT class** — TCGA AUC ≈ 0.40, F1 = 0.

| Model | CCLE val AUC | TCGA AUC | TCGA F1 | TCGA recall |
|---|---:|---:|---:|---:|
| GCN v2 thr=0.7 | 0.707 | **0.411** | 0.000 | 0.000 |
| GAT hybrid | 0.722 | **0.392** | 0.000 | 0.000 |

**Why does the GNN not transfer when XGBoost does?** Two compounding causes:

1. **BatchNorm running statistics** are fit on CCLE-distributed activations during training and applied unchanged at evaluation. TCGA activations live in a slightly different region → BN normalisation is wrong → distortion cascades through the network.
2. **Raw-expression node feature**: each node's input is `[raw_expression, z-score]`. The z-score component is per-cohort normalised and matches at inference time. The *raw* component, however, is in different absolute scales (CCLE log2(TPM+1) RSEM vs TCGA log2(norm_count+1) Xena), so part of the input is shifted relative to training.

Tree models tolerate this kind of moderate distribution shift because each split operates on a single feature at a single threshold; GNN weights are end-to-end and small distribution shifts cascade. This is a well-documented domain-shift failure mode and the principled fix is **explicit cross-cohort training**: domain-adversarial training (DANN), CORAL, LayerNorm in place of BatchNorm, or fine-tuning on a labelled TCGA subset. All flagged for future work.

---
## 5. Task 2 — Multiclass TP53 mutation type

### 5.1 Subset and class definitions

We restrict to the **986 TP53-mutant CCLE cell lines** and predict the mutation type. Frame_Shift_Del (n = 48) and Splice_Site (n = 65) are individually too small for stable 5-fold CV, so we merge them into a `Truncating` super-class (both functionally truncate the protein):

In [ ]:
dist = pd.read_csv(PROC / 'multiclass_class_distribution.csv')
dist.style.format({'fraction': '{:.1%}'})

Heavy class imbalance: Missense dominates at 64 %, Truncating is at 11 %. We use stratified 5-fold CV throughout.

### 5.2 Aggregate OOF results

In [ ]:
with open(PROC / 'multiclass_metrics.json') as f:
    m = json.load(f)
rows = []
for name in ['xgb', 'logreg']:
    a = m['models'][name]
    rows.append({
        'model': name.upper(),
        'accuracy': a['accuracy'],
        'macro_f1': a['macro_f1'],
        'weighted_f1': a['weighted_f1'],
        'roc_auc_ovr_macro': a['roc_auc_ovr_macro'],
    })
pd.DataFrame(rows).style.format({c: '{:.4f}' for c in
    ['accuracy', 'macro_f1', 'weighted_f1', 'roc_auc_ovr_macro']})

In [ ]:
pcm = pd.read_csv(PROC / 'multiclass_per_class_metrics.csv')
pcm.style.format({c: '{:.4f}' for c in ['precision', 'recall', 'f1']})

In [ ]:
display(Image(str(PLOTS / 'multiclass_per_class_f1.png')))

In [ ]:
display(Image(str(PLOTS / 'multiclass_confusion_xgb.png')))
display(Image(str(PLOTS / 'multiclass_confusion_logreg.png')))

**Two complementary failure modes**

- **XGBoost** achieves the higher accuracy (0.627) almost entirely by predicting the majority class — Missense recall = 0.97, but Truncating recall < 0.01 and Other recall = 0.03. This makes its accuracy look strong, but its **macro-F1 collapses to 0.28**.
- **Logistic Regression** (multinomial L2) is more honestly balanced — it actually distinguishes the minority classes (Truncating F1 = 0.17, Other F1 = 0.24), at the cost of some overall accuracy (0.54) but a meaningfully higher **macro-F1 of 0.37** and OvR macro AUC of 0.57.

**Biological reading** — TP53 mutation TYPE is fundamentally harder to predict from bulk transcriptome than mutation STATUS, because **all loss-of-function variants converge on the same downstream phenotype** (loss of p53 transcriptional activity, with similar collapse in p21 / apoptosis / DNA-repair targets). The remaining differences (e.g., dominant-negative effects of certain hotspot missense variants like R175H, R273H) are subtle and not robustly captured by bulk RNA-seq at our sample size. **This is itself an informative finding** — it confirms that the task-1 mutation-status signal is dominated by *whether* TP53 has been knocked out, not *how*.

### 5.3 Per-class top genes (XGBoost OvR feature importance)

Top genes per class (mean gain importance averaged across folds, from one-vs-rest XGBoost models):

In [ ]:
tg = pd.read_csv(PROC / 'top_genes_multiclass.csv')
for cls in ['Missense', 'Truncating', 'Other']:
    display(Markdown(f'**{cls}** — top 10:'))
    sub = tg[tg['class'] == cls].head(10)[['rank', 'gene_symbol', 'mean_xgb_importance']]
    display(sub.style.format({'mean_xgb_importance': '{:.4f}'}))

There is little overlap between class-specific top genes — each mutation type recruits a partially distinct expression signature for the discriminator, even though absolute discriminative power is low. None of the class-specific top genes is a canonical TP53 target (in contrast to the Task 1 SHAP top-20), suggesting that what little signal exists for mutation-type discrimination is in the *secondary* transcriptional fingerprint of each variant rather than the primary p53 regulome.

### 5.4 Notes on TCGA multiclass and GNN multiclass

**External multiclass validation on TCGA was deferred**. The MC3 binary mutation matrix gives only nonsilent / silent calls per gene; deriving harmonised mutation-type labels requires reading the TCGA MAF, parsing `Variant_Classification`, and reconciling annotation pipelines (CCLE uses VEP MolecularConsequence terms, TCGA MAF uses simplified VC categories). This is non-trivial and was out of scope for this iteration.

**GNN multiclass** was also deferred — given that GAT/GCN already trail XGBoost on the easier binary task and that mutation-type distinguishability is much weaker, a multiclass GNN would not change the qualitative finding. We document this as future work.

---
## 6. Discussion & conclusions

### Final scientific summary

**XGBoost on top-2,000 highly variable genes is the strongest model for predicting TP53 mutation status from bulk RNA-seq.** It matches the published Ravasio (2024) bulk benchmark within CCLE (F1 = 0.875, ROC-AUC = 0.906) and **transfers respectably to the TCGA pan-cancer atlas** (ROC-AUC = 0.806 with per-cohort z-score normalisation, > 0.85 on 8 of 31 cancer types after threshold calibration).

**SHAP analysis recovers canonical TP53 biology without supervision**. CDKN1A (p21) is the dominant feature with a mean |SHAP| three-fold larger than the next gene; formal hypergeometric testing confirms that the curated TP53 pathway is enriched among top-K SHAP genes at **52.6× (p = 1.07 × 10⁻⁸)** for K = 10. Several other top features (PHLDA3, BTG2, CYFIP2, FAS, TNFRSF10D) are documented direct TP53 transcriptional targets.

**GNNs improve substantially with better design** — the v2 architecture (BatchNorm, residual connections, balanced-class loss, early stopping, [raw, z-score] features) gains ~0.08 ROC-AUC over the vanilla GCN, and **GAT on a dense hybrid graph (STRING ∪ co-expression) reaches the highest GNN F1 (0.760)**. But **all GNN variants remain weaker than XGBoost** (gap of ≈ 0.12 F1, 0.20 ROC-AUC) on within-cohort metrics, **and they fail external transfer to TCGA** (AUC ≈ 0.40, model collapses) due to combined BatchNorm distribution shift and raw-feature scale mismatch. This is a real, well-documented domain-shift failure mode — not an indictment of GNNs but a clear signal that biological generalisation requires explicit cross-cohort training.

**Mutation TYPE prediction (Task 2) is much harder than mutation STATUS (Task 1)**, consistent with the loss-of-function biology — every TP53-mutant cell line converges on similar downstream transcriptional collapse, so the bulk transcriptome cannot easily distinguish how the gene was hit. Multinomial Logistic Regression achieves the best macro-F1 (0.37); XGBoost achieves higher accuracy but only by predicting the majority class.

### Methodological contributions

1. A clean head-to-head comparison of **statistical (Spearman) vs biological (STRING PPI) graph priors** for graph-based mutation prediction, demonstrating that the choice matters less than commonly assumed because the underlying signal is largely per-gene rather than relational.
2. A formal **hypergeometric test** showing that XGBoost feature importances are not just qualitatively but **quantitatively enriched** for the canonical TP53 pathway.
3. A documented **threshold calibration analysis** for cross-cohort transfer, separating discrimination quality from operating-point calibration.
4. An explicit **cross-cohort generalisation comparison** between tabular and graph models, including a clean negative finding for GNN transfer that points to specific architectural fixes.

## 7. Limitations & future work

**Limitations**

- All training is on the **DepMap 24Q4** snapshot. Re-training as new releases arrive would slightly shift specific results.
- Top-2k HVG was selected on the **full CCLE cohort** (not per-fold). For binary classification this is conservative because variance ranking is independent of TP53 labels, but a fully nested protocol would re-select per fold.
- TCGA labels are derived from the **MC3 binary nonsilent matrix**, which collapses heterogeneous mutation events into a single positive/negative call.
- The TCGA cohort is biased toward the ~30 cancer types in the PanCancer Atlas; rare cancers and paediatric tumours are not represented.
- The GNN cross-cohort failure was not actively mitigated (no domain-adaptation method tried in scope).

**Future work**

- **GNN domain adaptation** — DANN, CORAL, LayerNorm in place of BatchNorm, or fine-tuning on a labelled TCGA subset to recover cross-cohort transfer.
- **External multiclass validation** on TCGA — requires harmonising mutation-type labels from MAF VC categories.
- **Hyperparameter search** with Optuna for both XGBoost and GNN models — current results use fixed defaults.
- **TP53-target-only feature set** vs HVG comparison — does biology-driven feature selection match HVG performance with a fraction of the genes?
- **Multi-omics integration** — adding methylation, copy-number, or mutation matrix features.
- **Patient-level survival analysis** stratified by predicted TP53 status — the clinical actionability test.

## 8. References & data sources

- **CCLE / DepMap Public 24Q4** — Cancer Cell Line Encyclopedia. Broad Institute. [depmap.org](https://depmap.org/portal/).
- **TCGA PanCancer Atlas** — UCSC Xena hub. [pancanatlas.xenahubs.net](https://xenabrowser.net/datapages/?host=https://pancanatlas.xenahubs.net).
- **STRING v12.0** — physical protein-protein interaction network. [string-db.org](https://string-db.org/).
- **Ravasio, T. (2024)** — *Predicting TP53 Mutation Status from Single-Cell RNA-seq with Graph Neural Networks.* Bachelor thesis, Bocconi University. Bulk benchmark and methodological inspiration.
- **Vousden & Prives (2009)**; **Kastenhuber & Lowe (2017)** — TP53 pathway curation.
- **Computing**: Bocconi HPC cluster (`stud` partition, A100 GPUs).

Full methods, experimental log, and per-issue resolution documented in `PROJECT_NOTES.md` at the repository root.